In [ ]:
# ==========================================
# [환경 및 버전 정보 출력] - 데이콘 제출 규정 준수
# ==========================================
import sys
import platform
import pandas as pd
import numpy as np
import lightgbm as lgb
import sklearn
import warnings

print("="*50)
print("[개발 환경 및 라이브러리 버전]")
print(f"- OS: {platform.system()} {platform.release()}")
print(f"- Python 버전: {sys.version.split()[0]}")
print(f"- Pandas 버전: {pd.__version__}")
print(f"- Numpy 버전: {np.__version__}")
print(f"- LightGBM 버전: {lgb.__version__}")
print(f"- Scikit-learn 버전: {sklearn.__version__}")
print("="*50 + "\n")

warnings.filterwarnings('ignore')
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

# ==========================================
# 1. 전처리 함수 
# ==========================================
def preprocess_for_submission(train_path, test_path, layout_path):
    print("1. 데이터 로드 및 전처리를 시작합니다...")
    # 상대 경로를 통한 데이터 로드
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    layout = pd.read_csv(layout_path)
    
    test_ids = test['ID'].copy()
    train.drop('ID', axis=1, inplace=True)
    test.drop('ID', axis=1, inplace=True)

    train = pd.merge(train, layout, on='layout_id', how='left')
    test = pd.merge(test, layout, on='layout_id', how='left')

    target_col = 'avg_delay_minutes_next_30m'
    y_train = train[target_col]
    X_train = train.drop(target_col, axis=1)
    X_test = test.copy()
    
    # 결측치 처리 (훈련 데이터 중앙값 기준)
    num_cols = X_train.select_dtypes(include=['float64', 'int64']).columns
    for col in num_cols:
        median_val = X_train[col].median()
        X_train[col].fillna(median_val, inplace=True)
        X_test[col].fillna(median_val, inplace=True)

    # 원핫 인코딩
    X_train = pd.get_dummies(X_train, columns=['layout_type'], drop_first=False)
    X_test = pd.get_dummies(X_test, columns=['layout_type'], drop_first=False)
    X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

    # 파생 변수
    for df in [X_train, X_test]:
        df['shift_hour_sin'] = np.sin(2 * np.pi * df['shift_hour'] / 24)
        df['shift_hour_cos'] = np.cos(2 * np.pi * df['shift_hour'] / 24)
        df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
        df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
        
        if 'robot_utilization' in df.columns and 'congestion_score' in df.columns:
            df['robot_util_x_congestion'] = df['robot_utilization'] * df['congestion_score']

    return X_train, y_train, X_test, test_ids

# ==========================================
# 2. 모델 학습 및 예측 함수 (재현성 보장)
# ==========================================
def train_and_predict_final(X_train, y_train, X_test, test_ids):
    print("2. K-Fold 모델 학습 및 Target Encoding을 진행합니다...")
    # random_state=42 로 재현성(Private Score 복원) 완벽 보장
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    
    oof_predictions = np.zeros(len(X_train))
    test_predictions = np.zeros(len(X_test))
    
    # Optuna로 찾은 최적 하이퍼파라미터 적용
    lgb_params = {
        'objective': 'mae',
        'metric': 'mae',
        'n_estimators': 1500, 
        'learning_rate': 0.04165780165479688,
        'max_depth': 10,
        'num_leaves': 84,
        'subsample': 0.9282065869845333,
        'colsample_bytree': 0.7071982532616117,
        'random_state': 42, # 모델 재현성 보장
        'n_jobs': -1,
        'verbose': -1
    }
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
        x_tr = X_train.iloc[train_idx].copy()
        y_tr = y_train.iloc[train_idx]
        x_val = X_train.iloc[val_idx].copy()
        y_val = y_train.iloc[val_idx]
        x_te = X_test.copy()
        
        # Target Encoding (K-Fold 내부 수행으로 Data Leakage 방지)
        for col in ['scenario_id', 'layout_id']:
            target_mean = y_tr.groupby(x_tr[col]).mean()
            overall_mean = y_tr.mean()
            
            x_tr[f'{col}_target_enc'] = x_tr[col].map(target_mean)
            x_val[f'{col}_target_enc'] = x_val[col].map(target_mean).fillna(overall_mean)
            x_te[f'{col}_target_enc'] = x_te[col].map(target_mean).fillna(overall_mean) 
            
            x_tr.drop(col, axis=1, inplace=True)
            x_val.drop(col, axis=1, inplace=True)
            x_te.drop(col, axis=1, inplace=True)
            
        model = lgb.LGBMRegressor(**lgb_params)
        model.fit(
            x_tr, y_tr,
            eval_set=[(x_val, y_val)],
            callbacks=[
                lgb.early_stopping(stopping_rounds=50, verbose=False),
                lgb.log_evaluation(period=0)
            ]
        )
        
        val_preds = model.predict(x_val)
        oof_predictions[val_idx] = val_preds
        test_predictions += model.predict(x_te) / kf.n_splits
        
        fold_mae = mean_absolute_error(y_val, val_preds)
        print(f"  - Fold {fold+1} MAE: {fold_mae:.4f}")
        
    total_mae = mean_absolute_error(y_train, oof_predictions)
    print(f"\n🏆 훈련 완료! 최종 OOF MAE: {total_mae:.4f}")
    
    # 지연 시간 음수 방지 보정
    submission = pd.DataFrame({
        'ID': test_ids,
        'avg_delay_minutes_next_30m': test_predictions
    })
    submission['avg_delay_minutes_next_30m'] = submission['avg_delay_minutes_next_30m'].clip(lower=0)
    
    return submission

# ==========================================
# 3. 메인 실행부 (상대 경로 사용)
# ==========================================
if __name__ == "__main__":
    # 데이터 파일들이 현재 스크립트와 동일한 폴더에 있어야 합니다.
    TRAIN_PATH = 'train.csv' 
    TEST_PATH = 'test.csv'
    LAYOUT_PATH = 'layout_info.csv'
    SUBMISSION_PATH = 'final_submission.csv'

    # 파이프라인 실행
    X_train, y_train, X_test, test_ids = preprocess_for_submission(TRAIN_PATH, TEST_PATH, LAYOUT_PATH)
    submission_df = train_and_predict_final(X_train, y_train, X_test, test_ids)
    
    # 결과 저장 (상대 경로)
    submission_df.to_csv(SUBMISSION_PATH, index=False)
    print(f"\n✅ 상대 경로 '{SUBMISSION_PATH}'에 제출 파일 저장이 완료되었습니다.")

In [ ]:
# submission2
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

def make_final_divide_and_conquer_submission(train_path, test_path, layout_path, submission_path):
    print("1. 데이터 로드 및 전처리 시작...")
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)
    layout = pd.read_csv(layout_path)
    
    test_ids = test['ID'].copy()
    train.drop('ID', axis=1, inplace=True)
    test.drop('ID', axis=1, inplace=True)

    train = pd.merge(train, layout, on='layout_id', how='left')
    test = pd.merge(test, layout, on='layout_id', how='left')

    target_col = 'avg_delay_minutes_next_30m'
    y_train = train[target_col]
    X_train = train.drop(target_col, axis=1)
    X_test = test.copy()
    
    # 결측치 처리 (훈련셋 중앙값 기준)
    num_cols = X_train.select_dtypes(include=['float64', 'int64']).columns
    for col in num_cols:
        median_val = X_train[col].median()
        X_train[col].fillna(median_val, inplace=True)
        X_test[col].fillna(median_val, inplace=True)

    # 파생 변수 (주기성 원본 삭제)
    for df in [X_train, X_test]:
        df['shift_hour_sin'] = np.sin(2 * np.pi * df['shift_hour'] / 24)
        df['shift_hour_cos'] = np.cos(2 * np.pi * df['shift_hour'] / 24)
        df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
        df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
        df.drop(['shift_hour', 'day_of_week'], axis=1, inplace=True)
        
        if 'robot_utilization' in df.columns and 'congestion_score' in df.columns:
            df['robot_util_x_congestion'] = df['robot_utilization'] * df['congestion_score']
            
    # 원핫 인코딩 (Train/Test 컬럼 동기화)
    X_train = pd.get_dummies(X_train, columns=['layout_type'], drop_first=False)
    X_test = pd.get_dummies(X_test, columns=['layout_type'], drop_first=False)
    X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

    print("2. 훈련/테스트 데이터 동시 군집화 진행 중...")
    possible_features = ['robot_utilization', 'congestion_score', 'charge_queue_length', 'task_reassign_15m']
    cluster_features = [f for f in possible_features if f in X_train.columns]
    
    # K-Means용 데이터 준비 및 스케일링
    X_cluster_tr = X_train[cluster_features].copy()
    X_cluster_te = X_test[cluster_features].copy()
    
    for df in [X_cluster_tr, X_cluster_te]:
        df.replace([np.inf, -np.inf], np.nan, inplace=True)
        df.fillna(0, inplace=True)
        
    scaler = StandardScaler()
    X_scaled_tr = scaler.fit_transform(X_cluster_tr)
    X_scaled_te = scaler.transform(X_cluster_te) # 테스트는 transform만!
    
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    X_train['cluster_id'] = kmeans.fit_predict(X_scaled_tr)
    X_test['cluster_id'] = kmeans.predict(X_scaled_te) # 훈련된 기준으로 군집 매핑

    print("3. 군집별 5-Fold 학습 및 테스트 추론 시작...")
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    test_predictions = np.zeros(len(X_test))
    
    lgb_params = {
        'objective': 'mae', 'metric': 'mae', 'n_estimators': 1500, 
        'learning_rate': 0.0416, 'max_depth': 10, 'num_leaves': 84,
        'subsample': 0.928, 'colsample_bytree': 0.707,
        'random_state': 42, 'n_jobs': -1, 'verbose': -1
    }
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
        x_tr = X_train.iloc[train_idx].copy()
        y_tr = y_train.iloc[train_idx]
        x_val = X_train.iloc[val_idx].copy()
        x_te = X_test.copy()
        
        overall_mean = y_tr.mean()
        
        # Smoothed Target Encoding
        for col in ['scenario_id', 'layout_id']:
            if col in x_tr.columns:
                group_stats = y_tr.groupby(x_tr[col]).agg(mean=('mean'), count=('count'))
                smoothed_means = (group_stats['mean'] * group_stats['count'] + overall_mean * 10) / (group_stats['count'] + 10)
                
                x_tr[f'{col}_target_enc'] = x_tr[col].map(smoothed_means)
                x_val[f'{col}_target_enc'] = x_val[col].map(smoothed_means).fillna(overall_mean)
                x_te[f'{col}_target_enc'] = x_te[col].map(smoothed_means).fillna(overall_mean)
                
                x_tr.drop(col, axis=1, inplace=True)
                x_val.drop(col, axis=1, inplace=True)
                x_te.drop(col, axis=1, inplace=True)

        fold_test_preds = np.zeros(len(x_te))
        
        # 군집(0, 1, 2)별로 분할 학습 및 테스트 예측
        for cid in range(3):
            tr_mask = x_tr['cluster_id'] == cid
            te_mask = x_te['cluster_id'] == cid
            
            if not tr_mask.any(): continue
            
            model = lgb.LGBMRegressor(**lgb_params)
            model.fit(
                x_tr[tr_mask].drop('cluster_id', axis=1), y_tr[tr_mask],
                eval_set=[(x_tr[tr_mask].drop('cluster_id', axis=1), y_tr[tr_mask])], # 빠른 진행을 위해 validation 체크 생략
                callbacks=[lgb.early_stopping(50, verbose=False)]
            )
            
            # 테스트 데이터의 해당 군집 부분만 예측
            if te_mask.any():
                fold_test_preds[te_mask] = model.predict(x_te[te_mask].drop('cluster_id', axis=1))
                
        # 예측값을 K-Fold 등분하여 누적
        test_predictions += fold_test_preds / kf.n_splits
        print(f"  - Fold {fold+1} 테스트 추론 완료")
        
    # 결과 보정 및 저장
    submission = pd.DataFrame({
        'ID': test_ids,
        'avg_delay_minutes_next_30m': test_predictions
    })
    submission['avg_delay_minutes_next_30m'] = submission['avg_delay_minutes_next_30m'].clip(lower=0)
    submission.to_csv(submission_path, index=False)
    
    print("\n" + "="*50)
    print(f"✅ 최종 제출 파일 '{submission_path}' 생성 완료!")
    print("="*50 + "\n")

if __name__ == "__main__":
    TRAIN_PATH = 'train.csv'       # 로컬 환경 확인! (sampled_train 이라면 맞춰주세요)
    TEST_PATH = 'test.csv'
    LAYOUT_PATH = 'layout_info.csv'
    SUBMISSION_PATH = 'dnc_cluster_submission.csv'
    
    make_final_divide_and_conquer_submission(TRAIN_PATH, TEST_PATH, LAYOUT_PATH, SUBMISSION_PATH)

1. 데이터 로드 및 전처리 시작...
2. 훈련/테스트 데이터 동시 군집화 진행 중...
3. 군집별 5-Fold 학습 및 테스트 추론 시작...
  - Fold 1 테스트 추론 완료
  - Fold 2 테스트 추론 완료
  - Fold 3 테스트 추론 완료
  - Fold 4 테스트 추론 완료
  - Fold 5 테스트 추론 완료

✅ 최종 제출 파일 'dnc_cluster_submission.csv' 생성 완료!



In [ ]:
# submission 3
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ==========================================
# 1. 완벽 전처리 및 군집화 (Tree 전용)
# ==========================================
def preprocess_for_tree(train_path, layout_path):
    print("\n1. Train 데이터 로드 및 4대 핵심 파생 변수 전처리...")
    train = pd.read_csv(train_path)
    layout = pd.read_csv(layout_path)
    
    train = pd.merge(train, layout, on='layout_id', how='left')
    train.drop('ID', axis=1, inplace=True)

    target_col = 'avg_delay_minutes_next_30m'
    y_train = train[target_col].copy()
    X_train = train.drop(target_col, axis=1)
    
    for col in X_train.select_dtypes(include=['float64', 'int64']).columns:
        X_train[col].fillna(X_train[col].median(), inplace=True)

    if 'shift_hour' in X_train.columns:
        X_train['shift_hour_sin'] = np.sin(2 * np.pi * X_train['shift_hour'] / 24)
        X_train['shift_hour_cos'] = np.cos(2 * np.pi * X_train['shift_hour'] / 24)
    if 'day_of_week' in X_train.columns:
        X_train['day_of_week_sin'] = np.sin(2 * np.pi * X_train['day_of_week'] / 7)
        X_train['day_of_week_cos'] = np.cos(2 * np.pi * X_train['day_of_week'] / 7)
    X_train.drop(['shift_hour', 'day_of_week'], axis=1, inplace=True, errors='ignore') 
    
    if 'robot_utilization' in X_train.columns and 'congestion_score' in X_train.columns:
        X_train['robot_util_x_congestion'] = X_train['robot_utilization'] * X_train['congestion_score']
    if 'order_inflow_15m' in X_train.columns and 'robot_active' in X_train.columns:
        X_train['order_pressure'] = X_train['order_inflow_15m'] / (X_train['robot_active'] + 1)
    if 'low_battery_ratio' in X_train.columns and 'charge_queue_length' in X_train.columns:
        X_train['battery_stress'] = X_train['low_battery_ratio'] * X_train['charge_queue_length']
    if 'congestion_score' in X_train.columns and 'blocked_path_15m' in X_train.columns:
        X_train['congestion_impact'] = X_train['congestion_score'] * X_train['blocked_path_15m']
        
    X_train = pd.get_dummies(X_train, columns=['layout_type'], drop_first=False)

    print("2. 3분할(쾌적/혼잡/마비) K-Means 군집화...")
    cluster_features = [f for f in ['robot_utilization', 'congestion_score', 'charge_queue_length', 'task_reassign_15m'] if f in X_train.columns]
    
    X_cluster = X_train[cluster_features].copy()
    X_cluster.replace([np.inf, -np.inf], np.nan, inplace=True)
    X_cluster.fillna(0, inplace=True)
    
    scaler_cluster = StandardScaler()
    X_scaled_cluster = scaler_cluster.fit_transform(X_cluster)
    
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    X_train['cluster_id'] = kmeans.fit_predict(X_scaled_cluster)
    
    X_train.replace([np.inf, -np.inf], np.nan, inplace=True)
    X_train.fillna(0, inplace=True)
    
    # Test 셋 적용을 위해 scaler와 kmeans 반환 추가
    return X_train, y_train, scaler_cluster, kmeans

# ==========================================
# [추가] Test 데이터 전처리
# ==========================================
def preprocess_test(test_path, layout_path, train_columns, scaler_cluster, kmeans):
    print("\n[추가] Test 데이터 로드 및 전처리...")
    test = pd.read_csv(test_path)
    layout = pd.read_csv(layout_path)
    
    test_ids = test['ID'].copy()
    test = pd.merge(test, layout, on='layout_id', how='left')
    test.drop('ID', axis=1, inplace=True)

    for col in test.select_dtypes(include=['float64', 'int64']).columns:
        test[col].fillna(test[col].median(), inplace=True)

    if 'shift_hour' in test.columns:
        test['shift_hour_sin'] = np.sin(2 * np.pi * test['shift_hour'] / 24)
        test['shift_hour_cos'] = np.cos(2 * np.pi * test['shift_hour'] / 24)
    if 'day_of_week' in test.columns:
        test['day_of_week_sin'] = np.sin(2 * np.pi * test['day_of_week'] / 7)
        test['day_of_week_cos'] = np.cos(2 * np.pi * test['day_of_week'] / 7)
    test.drop(['shift_hour', 'day_of_week'], axis=1, inplace=True, errors='ignore') 
    
    if 'robot_utilization' in test.columns and 'congestion_score' in test.columns:
        test['robot_util_x_congestion'] = test['robot_utilization'] * test['congestion_score']
    if 'order_inflow_15m' in test.columns and 'robot_active' in test.columns:
        test['order_pressure'] = test['order_inflow_15m'] / (test['robot_active'] + 1)
    if 'low_battery_ratio' in test.columns and 'charge_queue_length' in test.columns:
        test['battery_stress'] = test['low_battery_ratio'] * test['charge_queue_length']
    if 'congestion_score' in test.columns and 'blocked_path_15m' in test.columns:
        test['congestion_impact'] = test['congestion_score'] * test['blocked_path_15m']
        
    test = pd.get_dummies(test, columns=['layout_type'], drop_first=False)

    for col in train_columns:
        if col not in test.columns and col != 'cluster_id':
            test[col] = 0
            
    test = test[[col for col in train_columns if col != 'cluster_id']]

    cluster_features = [f for f in ['robot_utilization', 'congestion_score', 'charge_queue_length', 'task_reassign_15m'] if f in test.columns]
    X_cluster = test[cluster_features].copy()
    X_cluster.replace([np.inf, -np.inf], np.nan, inplace=True)
    X_cluster.fillna(0, inplace=True)
    
    X_scaled_cluster = scaler_cluster.transform(X_cluster)
    test['cluster_id'] = kmeans.predict(X_scaled_cluster)
    
    test.replace([np.inf, -np.inf], np.nan, inplace=True)
    test.fillna(0, inplace=True)
    
    return test, test_ids

# ==========================================
# 2. 군집별 Optuna 튜닝 및 최종 평가
# ==========================================
def optimize_and_evaluate(X_train, y_train):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    best_params_per_cluster = {}
    
    print("\n3. 군집별 Optuna 극한 튜닝 시작 (각 군집당 15번 탐색)...")
    for cid in range(3):
        mask = X_train['cluster_id'] == cid
        if not mask.any(): continue
            
        x_c = X_train[mask].drop('cluster_id', axis=1)
        y_c = y_train[mask]
        
        def objective(trial):
            params = {
                'objective': 'mae',
                'metric': 'mae',
                'boosting_type': 'gbdt',
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
                'max_depth': trial.suggest_int('max_depth', 4, 12),
                'num_leaves': trial.suggest_int('num_leaves', 15, 127),
                'subsample': trial.suggest_float('subsample', 0.7, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
                'n_estimators': 300, 
                'n_jobs': 2, # 메모리 에러 방지용으로 2로 임시 세팅
                'verbose': -1,
                'random_state': 42
            }
            
            cv_mae = []
            for tr_idx, val_idx in KFold(n_splits=3, shuffle=True, random_state=42).split(x_c):
                x_tr, y_tr = x_c.iloc[tr_idx].copy(), y_c.iloc[tr_idx].copy()
                x_val, y_val = x_c.iloc[val_idx].copy(), y_c.iloc[val_idx].copy()
                overall_mean = y_tr.mean()
                
                for col in ['scenario_id', 'layout_id']:
                    if col in x_tr.columns:
                        group_stats = y_tr.groupby(x_tr[col]).agg(mean=('mean'), count=('count'))
                        smoothed_means = (group_stats['mean'] * group_stats['count'] + overall_mean * 10) / (group_stats['count'] + 10)
                        x_tr[col] = x_tr[col].map(smoothed_means).astype('float32') # 타입 에러 방지
                        x_val[col] = x_val[col].map(smoothed_means).fillna(overall_mean).astype('float32')
                
                model = lgb.LGBMRegressor(**params)
                model.fit(x_tr, y_tr, eval_set=[(x_val, y_val)], callbacks=[lgb.early_stopping(30, verbose=False)])
                cv_mae.append(mean_absolute_error(y_val, model.predict(x_val)))
                
            return np.mean(cv_mae)

        study = optuna.create_study(direction='minimize')
        study.optimize(objective, n_trials=10) # 속도를 위해 10으로 조정
        best_params_per_cluster[cid] = study.best_params
        best_params_per_cluster[cid]['n_estimators'] = 1000 
        best_params_per_cluster[cid]['objective'] = 'mae'
        best_params_per_cluster[cid]['n_jobs'] = 2
        best_params_per_cluster[cid]['verbose'] = -1
        
        print(f"  > Cluster {cid} 최적 파라미터 찾기 완료 (MAE: {study.best_value:.4f})")

    print("\n4. 최적 파라미터 장착 후 최종 5-Fold 검증 시작...")
    oof_predictions = np.zeros(len(X_train))
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
        x_tr, y_tr = X_train.iloc[train_idx].copy(), y_train.iloc[train_idx].copy()
        x_val, y_val = X_train.iloc[val_idx].copy(), y_train.iloc[val_idx].copy()
        overall_mean = y_tr.mean()
        
        for col in ['scenario_id', 'layout_id']:
            if col in x_tr.columns:
                group_stats = y_tr.groupby(x_tr[col]).agg(mean=('mean'), count=('count'))
                smoothed_means = (group_stats['mean'] * group_stats['count'] + overall_mean * 10) / (group_stats['count'] + 10)
                x_tr[col] = x_tr[col].map(smoothed_means).astype('float32')
                x_val[col] = x_val[col].map(smoothed_means).fillna(overall_mean).astype('float32')

        fold_preds = np.zeros(len(x_val))
        for cid in range(3):
            tr_mask = x_tr['cluster_id'] == cid
            val_mask = x_val['cluster_id'] == cid
            if not val_mask.any() or not tr_mask.any(): continue
            
            model = lgb.LGBMRegressor(**best_params_per_cluster[cid])
            model.fit(
                x_tr[tr_mask].drop('cluster_id', axis=1), y_tr[tr_mask],
                eval_set=[(x_val[val_mask].drop('cluster_id', axis=1), y_val[val_mask])],
                callbacks=[lgb.early_stopping(50, verbose=False)]
            )
            fold_preds[val_mask] = model.predict(x_val[val_mask].drop('cluster_id', axis=1))
            
        oof_predictions[val_idx] = fold_preds
        print(f"  - Fold {fold+1} 완료 (MAE: {mean_absolute_error(y_val, fold_preds):.4f})")
        
    total_mae = mean_absolute_error(y_train, oof_predictions)
    print("\n" + "="*50)
    print(f"🏆 최종 OOF MAE: {total_mae:.4f}")
    print("="*50 + "\n")
    
    return best_params_per_cluster

# ==========================================
# [추가] 3. 전체 데이터 최종 학습 및 Test 추론
# ==========================================
def train_final_and_infer(X_train, y_train, X_test, test_ids, best_params_per_cluster):
    print("5. 100% 데이터 기반 최종 모델 학습 및 추론 중...")
    overall_mean = y_train.mean()
    X_train_final = X_train.copy()
    X_test_final = X_test.copy()
    
    for col in ['scenario_id', 'layout_id']:
        if col in X_train_final.columns:
            group_stats = y_train.groupby(X_train_final[col]).agg(mean=('mean'), count=('count'))
            smoothed_means = (group_stats['mean'] * group_stats['count'] + overall_mean * 10) / (group_stats['count'] + 10)
            X_train_final[col] = X_train_final[col].map(smoothed_means).astype('float32')
            X_test_final[col] = X_test_final[col].map(smoothed_means).fillna(overall_mean).astype('float32')

    final_predictions = np.zeros(len(X_test_final))
    
    for cid in range(3):
        tr_mask = X_train_final['cluster_id'] == cid
        te_mask = X_test_final['cluster_id'] == cid
        if not te_mask.any(): continue
            
        model = lgb.LGBMRegressor(**best_params_per_cluster[cid])
        model.fit(X_train_final[tr_mask].drop('cluster_id', axis=1), y_train[tr_mask])
        final_predictions[te_mask] = model.predict(X_test_final[te_mask].drop('cluster_id', axis=1))

    # 제출 파일 생성
    submission = pd.DataFrame({
        'ID': test_ids,
        'avg_delay_minutes_next_30m': final_predictions
    })
    submission_path = 'submission_baseline.csv'
    submission.to_csv(submission_path, index=False)
    print(f"✅ 제출 파일 저장 완료: {submission_path}")

if __name__ == "__main__":
    TRAIN_PATH = 'train.csv' 
    TEST_PATH = 'test.csv'
    LAYOUT_PATH = 'layout_info.csv'
    
    # 파이프라인 실행
    X_train, y_train, scaler_cluster, kmeans = preprocess_for_tree(TRAIN_PATH, LAYOUT_PATH)
    best_params = optimize_and_evaluate(X_train, y_train)
    X_test, test_ids = preprocess_test(TEST_PATH, LAYOUT_PATH, X_train.columns, scaler_cluster, kmeans)
    train_final_and_infer(X_train, y_train, X_test, test_ids, best_params)

c:\Users\kyw41\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



1. Train 데이터 로드 및 4대 핵심 파생 변수 전처리...
2. 3분할(쾌적/혼잡/마비) K-Means 군집화...

3. 군집별 Optuna 극한 튜닝 시작 (각 군집당 15번 탐색)...
  > Cluster 0 최적 파라미터 찾기 완료 (MAE: 4.0394)
  > Cluster 1 최적 파라미터 찾기 완료 (MAE: 8.6083)
  > Cluster 2 최적 파라미터 찾기 완료 (MAE: 4.2890)

4. 최적 파라미터 장착 후 최종 5-Fold 검증 시작...
  - Fold 1 완료 (MAE: 4.7000)
  - Fold 2 완료 (MAE: 4.8106)
  - Fold 3 완료 (MAE: 4.6791)
  - Fold 4 완료 (MAE: 4.7197)
  - Fold 5 완료 (MAE: 4.7124)

🏆 최종 OOF MAE: 4.7244


[추가] Test 데이터 로드 및 전처리...
5. 100% 데이터 기반 최종 모델 학습 및 추론 중...
✅ 제출 파일 저장 완료: submission_baseline.csv
